# Foundation-Sec-8B QLoRA — Kaggle (free T4)

Fine-tunes **fdtn-ai/Foundation-Sec-8B-Instruct** on `data/train.jsonl`, then exports a
merged **GGUF (Q4_K_M)**; `make_modelfile.py` then writes the Ollama
Modelfile on your machine.

**Before running:** enable *GPU T4* and *Internet* in the Kaggle sidebar, and add `data/`
as a Kaggle Dataset input. Run with `SMOKE = True` once to validate the whole path.

In [ ]:
SMOKE = True            # True = 50 rows / 10 steps (validate); False = full run
BASE_MODEL = "fdtn-ai/Foundation-Sec-8B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 42
OUT_GGUF = "gguf_model"

In [ ]:
# Kaggle internet must be ON (sidebar). Unsloth pulls transformers/trl/peft/bitsandbytes.
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
# Copy the prep notebook's output (attached via Add Input -> /kaggle/input/...) into ./data
import glob, os, shutil
os.makedirs("data", exist_ok=True)
for name in ("train.jsonl", "eval.jsonl", "system_prompt.txt"):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if hits:
        shutil.copy(hits[0], f"data/{name}")
    assert os.path.exists(f"data/{name}"), f"{name} not found: attach the prep notebook's output"
print(sorted(os.listdir("data")))

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "eval": "data/eval.jsonl"})

def to_text(ex):
    # Rows are already in Foundation-Sec's own format (system/user/assistant, tool calls
    # as <tool_call> text), so the model's chat template renders them as-is.
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    return {"text": text.removeprefix(tokenizer.bos_token or "")}  # tokenizer adds BOS itself

ds = ds.map(to_text, remove_columns=ds["train"].column_names)  # keep only "text"
if SMOKE:
    ds["train"] = ds["train"].select(range(min(50, len(ds["train"]))))
print(ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir="out", dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    num_train_epochs=1, max_steps=(10 if SMOKE else -1),
    logging_steps=5, optim="paged_adamw_8bit", seed=SEED, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"], args=args)
trainer.train()

In [ ]:
# Quick held-out sanity check (not a full eval — see DATA_PLAN.md).
print(trainer.evaluate(eval_dataset=ds["eval"]))

## Export — merge BEFORE GGUF

Unsloth issue **#11698**: exporting to GGUF *without merging first* silently ships the
base model and throws your fine-tune away. So we merge explicitly, then quantize. On a
16 GB T4 the fp16 merge is tight — Kaggle's ~30 GB system RAM absorbs it.

In [ ]:
# 1) merge the LoRA adapter into the base weights (explicit, before any export)
model.save_pretrained_merged("merged_16bit", tokenizer, save_method="merged_16bit")

# 2) quantize the merged model to GGUF Q4_K_M (fits an 8 GB RTX 4060 locally)
model.save_pretrained_gguf(OUT_GGUF, tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob
print("GGUF:", sorted(glob.glob(f"{OUT_GGUF}/*.gguf")))
# Download that .gguf from the Output panel. Then on your own machine, in the cyber-llm repo:
#   python3 make_modelfile.py ~/Downloads/<file>.gguf
#   ollama create cyber-sec -f ~/Downloads/Modelfile && ollama run cyber-sec
# Ignore any Modelfile Unsloth wrote here; make_modelfile.py matches the training format.

Only the ~5 GB GGUF comes down to your machine. Everything above ran on Kaggle's network.